# VNAT Transfer Learning Verification

This notebook verifies the performance of the HMVCL model on the VNAT dataset using a **Transfer Learning** protocol (often referred to as 'Zero-Shot Domain Adaptation' in the paper, though technically involving fine-tuning the classifier head).

**Protocol:**
1. Load Pre-trained Encoder (trained on ISCX-VPN-2016).
2. Freeze Encoder -> Extract Features from VNAT.
3. Train XGBoost Classifier on VNAT subsets (5%, 10%, 20%, 30%).
4. Evaluate Binary Detection (VPN vs Non-VPN).

In [11]:
import tensorflow as tf
from tensorflow.keras import models, Model, layers
import numpy as np
import pandas as pd
import os
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_sample_weight

# --- CONFIGURATION ---
BASE_PATH = "/content/drive/MyDrive/1 Skripsi/27jan/"

# VNAT Input Data
VNAT_VIEW1_PATH = os.path.join(BASE_PATH, "VNAT_v3PRETRAIN_X_view1.npy") # (N, 10, 784)
VNAT_VIEW2_PATH = os.path.join(BASE_PATH, "VNAT_v3PRETRAIN_X_view2.npy") # (N, 137)
VNAT_CSV_PATH = os.path.join(BASE_PATH, "VNAT_merged_components_consistent.csv")

# Pre-trained Weights (from ISCX)
ENCODER_WEIGHTS_PATH = os.path.join(BASE_PATH, "v3FULL_HMVCL_Encoder.weights.h5")

# Experiment Settings
LABEL_PERCENTAGES = [0.05, 0.10, 0.20, 0.30]
LATENT_DIM = 128

# --- 1. Architecture (Must Validly Match Pre-training) ---
def get_cnn_encoder(input_shape=(10, 784)):
    inputs = layers.Input(shape=input_shape)
    x = layers.Reshape((input_shape[0] * input_shape[1], 1))(inputs)
    
    # Standard CNN Block
    x = layers.Conv1D(32, 7, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(4)(x)
    x = layers.Conv1D(64, 5, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(4)(x)
    x = layers.Conv1D(128, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(4)(x)
    
    x = layers.Flatten()(x)
    h = layers.Dense(LATENT_DIM, activation='relu', name="representation")(x)
    z = layers.Dense(64, activation='relu', name="projection")(h)
    
    return Model(inputs, [h, z], name="CNN_Encoder")

# --- 2. Data Loading ---
def load_vnat_data():
    print("--- Loading VNAT Data ---")
    if not os.path.exists(VNAT_VIEW1_PATH):
        # Fallback check for local testing if needed, but per instruction using hardcoded BASE_PATH
        pass 

    try:
        X_v1 = np.load(VNAT_VIEW1_PATH).astype('float32')
        X_v2 = np.load(VNAT_VIEW2_PATH).astype('float32')
        df = pd.read_csv(VNAT_CSV_PATH)
    except Exception as e:
        print(f"Data Load Error: {e}")
        return None, None, None

    # Normalize Stats (Using own stats, Domain Adaptation usually requires re-norm)
    X_v2 = (X_v2 - np.mean(X_v2, axis=0)) / (np.std(X_v2, axis=0) + 1e-8)
    
    return df, X_v1, X_v2

# --- 3. Feature Extraction ---
def extract_features(X_v1, weights_path):
    print(f"Loading Encoder Weights: {weights_path}")
    cnn = get_cnn_encoder()
    try:
        cnn.load_weights(weights_path)
        print("Weights loaded successfully.")
    except Exception as e:
        print(f"Error loading weights: {e}")
        return None

    # Extract 'h'
    extractor = Model(inputs=cnn.input, outputs=cnn.outputs[0])
    return extractor.predict(X_v1, batch_size=128, verbose=1)

# --- 4. Main ---
def main():
    # 1. Load
    df, X_pay, X_stat = load_vnat_data()
    if df is None: return

    # 2. Extract
    X_deep = extract_features(X_pay, ENCODER_WEIGHTS_PATH)
    if X_deep is None: return

    # 3. Fuse
    X_final = np.concatenate([X_stat, X_deep], axis=1)
    print(f"Fused Features Shape: {X_final.shape}")

    # 4. Prepare Labels (Binary Only for VNAT)
    labels = []
    if 'filename' not in df.columns:
        print("Error: 'filename' column missing in CSV")
        return

    for fname in df['filename']:
        f_lower = str(fname).lower()
        # Robust check
        if "non-vpn" in f_lower or "nonvpn" in f_lower:
            labels.append("Non-VPN")
        elif "vpn" in f_lower:
            labels.append("VPN")
        else:
            # Fallback (non-vpn usually covers everything else in VNAT)
            labels.append("Non-VPN")
    
    y = LabelEncoder().fit_transform(labels)
    unique_classes = np.unique(y)
    print(f"Class Distribution: {np.bincount(y)}")
    if len(unique_classes) < 2:
        print(f"Error: Only {len(unique_classes)} class detected ({np.unique(labels)}). XGBoost requires >= 2 classes.")
        print("Sample Filenames:", df['filename'].head(10).tolist())
        return

    # 5. Run Experiments
    print("\n=== VNAT Experiment Results ===")
    print(f"{'Label %':<10} | {'Macro F1':<10} | {'Support (Test)':<15}")
    print("-"*45)

    for pct in LABEL_PERCENTAGES:
        try:
            X_train, X_test, y_train, y_test = train_test_split(
                X_final, y, train_size=pct, stratify=y, random_state=42
            )
            
            weights = compute_sample_weight('balanced', y_train)
            clf = xgb.XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.05, n_jobs=-1, eval_metric='logloss')
            clf.fit(X_train, y_train, sample_weight=weights)
            
            y_pred = clf.predict(X_test)
            f1 = f1_score(y_test, y_pred, average='macro')
            
            print(f"{pct*100:<9.0f}% | {f1:.4f}     | {len(y_test)}")
        except Exception as e:
            print(f"{pct*100:<9.0f}% | ERROR: {e}")
    
    print("\nDone.")

if __name__ == "__main__":
    main()


--- Loading VNAT Data ---
Loading Encoder Weights: /content/drive/MyDrive/1 Skripsi/27jan/v3FULL_HMVCL_Encoder.weights.h5
Weights loaded successfully.
29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step
Fused Features Shape: (3709, 265)
Class Distribution: [3626   83]

=== VNAT Experiment Results ===
Label %    | Macro F1   | Support (Test) 
---------------------------------------------
5        % | 0.9833     | 3524
10       % | 0.9824     | 3339
20       % | 0.9799     | 2968
30       % | 0.9770     | 2597

Done.
